# VietMedBridge — 05: Hard negatives → QLoRA → dev F2 chọn checkpoint

Giai đoạn supervised trong master §§13/27/39/56. **04 chạy pretrained và
submit được ngay**, không cần chạy 05 trước khi có nhãn. Chọn runtime GPU
mới; L4/A100 thuận tiện hơn cho training. Qwen8B nạp NF4, batch=1,
gradient checkpointing/accumulation.

Hai file trong cùng Drive: data/labels/retrieval_train.json và
data/labels/retrieval_dev.json. Schema: {reviewed:true, split:"train" hoặc
"dev", exhaustive_chunks:true, queries:[{id:..., query:"...",
relevant_docs:[...], relevant_chunks:[{doc_id:...,chunk_text:"source đã review"}],
negative_categories:{child_id:"same_document_wrong_chunk|same_disease_wrong_intervention|same_entity_wrong_context|cross_language_false_friend|high_lexical_overlap"}}]}.
Khi gold không exhaustive, chỉ dùng negative_child_ids đã review. Category
thiếu giữ unclassified; không giả vờ đã kiểm semantic taxonomy.

Train/dev tách theo query, IDs và nội dung không trùng 1.200 contest queries.
Mine 1 positive + 7 negatives, ưu tiên top ranks, không lấy predictions làm gold.
QLoRA checkpoint giữ optimizer/RNG để resume. Đánh giá dev NDCG/MRR rồi
chạy finalists qua full pipeline và CPU F2 cutoff sweep. Chọn adapter theo
dev F2, giữ trạng thái held-out pending; không promote corpus.
04 tự nhận selected_adapter/calibrated_policy trong namespace mới.


## 1. Bootstrap — CPU


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "chọn phiên bản runtime có Python trong khoảng này."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
# Chạy lại cell này sau MỖI runtime mới/restart: package chỉ được cài trong phiên Colab.
drive.mount("/content/drive")

# Giữ DATA_ROOT đã dùng ở notebook 00–03.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
EXPECTED_PIPELINE_API = 2
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "training_code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
if lock and lock.get("pipeline_api") != EXPECTED_PIPELINE_API and not CODE_REVISION:
    raise RuntimeError(
        "DATA_ROOT đang dùng code cũ. Để nâng lên data-v2, đặt CODE_REVISION='main' "
        "ở cell này rồi chạy lại; dùng tên crawl/build run mới. Raw cũ vẫn được giữ."
    )
RETRIEVAL_WORKFLOW_API = "full-master-plan-strong-v1"
upgrade = lock.get("workflow_api") != RETRIEVAL_WORKFLOW_API
reference = CODE_REVISION or ("main" if upgrade else lock.get("git_commit")) or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if "vietmedbridge" in sys.modules and globals().get("_VMB_IMPORTED_COMMIT") != CODE_COMMIT:
    raise RuntimeError("Mã nguồn đổi trong runtime đã import package. Restart session rồi chạy lại notebook.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-e", ".[notebook,retrieval,strong]"],
    cwd=CHECKOUT,
    check=True,
)
PACKAGE_SOURCE = CHECKOUT / "src"
if not (PACKAGE_SOURCE / "vietmedbridge" / "__init__.py").is_file():
    raise RuntimeError(
        f"Repo checkout thiếu package source: {PACKAGE_SOURCE}; commit={CODE_COMMIT}. "
        "Kiểm tra lại git fetch/checkout ở output của cell Bootstrap."
    )
sys.path.insert(0, str(PACKAGE_SOURCE))
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json, runtime_versions
from vietmedbridge import PIPELINE_API_VERSION
if PIPELINE_API_VERSION != EXPECTED_PIPELINE_API:
    raise RuntimeError("Notebook và package API không khớp; chọn commit data-v2 và restart session.")
_VMB_IMPORTED_COMMIT = CODE_COMMIT
if not lock or CODE_REVISION or upgrade:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT, "pipeline_api": PIPELINE_API_VERSION, "workflow_api": RETRIEVAL_WORKFLOW_API})
PIPELINE_CONFIG = json.loads((CHECKOUT / "configs/data_pipeline.json").read_text())
atomic_json(DATA_ROOT / "training_runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
import vietmedbridge
print("Package source:", Path(vietmedbridge.__file__).resolve())
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 2. Chạy supervised workflow — GPU khi đã có nhãn hợp lệ


In [ ]:
import json
from vietmedbridge.training_workflow import run_training_workflow
TRAINING_STATUS = run_training_workflow(DATA_ROOT, CHECKOUT,
    work_dir=WORK_DIR, run_name="stage-a-qlora-v1")
print(json.dumps(TRAINING_STATUS, ensure_ascii=False, indent=2))


Không có nhãn: WAITING_FOR_INDEPENDENT_REVIEWED_TRAIN_DEV_LABELS, không tải
model để giả lập training. Gold/negative pool thiếu: giữ mining holds để
review. Ablation/embedding fine-tune/sharded dense có API trong package,
hướng dẫn ở docs/FULL_PLAN.md. Sau khi chọn adapter hợp lệ, mở 04 và Run all;
không chỉnh ACTION hoặc xóa run pretrained. Nhãn/config đổi cần run mới.
